# Citation Network Analysis

Constructs and analyses the citation network for the chess research corpus using two data sources:

- **Crossref** – reference lists fetched during metadata enrichment
- **OpenAlex** – richer reference resolution, year backfill, and full reference graph

Cells run in order. Each section writes intermediate files so steps can be re-run independently.

**File pipeline:**
`metadata_enriched_FINAL.csv` → `metadata_enriched_after_openalex_doi.csv` → `metadata_enriched_after_openalex_year.csv`

---

## 1. Baseline Statistics & Crossref Citation Graph

Quick sanity checks on DOI/reference coverage, then builds a Crossref-based directed citation graph and a bibliographic coupling graph.

In [3]:
import ast
import json
import math
import re
import time
from collections import defaultdict
from difflib import SequenceMatcher
from itertools import combinations
from pathlib import Path
from typing import List, Optional

import networkx as nx
import pandas as pd
import requests
from pyvis.network import Network
from rapidfuzz import fuzz

OUTPUT_DIR       = Path("..").resolve() / "output"
META_FINAL_CSV   = OUTPUT_DIR / "metadata_enriched_FINAL.csv"
META_OA_DOI_CSV  = OUTPUT_DIR / "metadata_enriched_after_openalex_doi.csv"
META_OA_YEAR_CSV = OUTPUT_DIR / "metadata_enriched_after_openalex_year.csv"
CACHE_PATH       = OUTPUT_DIR / "openalex_citations_cache.json"
GRAPHML_OUT      = OUTPUT_DIR / "articles_citations.graphml"
HTML_OUT         = OUTPUT_DIR / "articles_citations.html"
GRAPHML_ERA_OUT  = OUTPUT_DIR / "articles_citations_by_era.graphml"
HTML_ERA_OUT     = OUTPUT_DIR / "articles_citations_by_era.html"

TITLE_SIM_THRESHOLD  = 0.90
YEAR_SLACK           = 1
SEARCH_PER_PAGE      = 5
REQUEST_SLEEP_S      = 0.12
TITLE_SEARCH_MAX_LEN = 350
BATCH_SIZE           = 50

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "leno4ka-doi-resolver/1", "Accept": "application/json"})

df = pd.read_csv(META_FINAL_CSV)

print(f"Total articles: {len(df)}")
print(f"With DOI: {df['doi'].notna().sum()} ({df['doi'].notna().mean()*100:.1f}%)")
print(f"With references: {df['crossref_references'].notna().sum()} ({df['crossref_references'].notna().mean()*100:.1f}%)")

# Build directed citation graph from Crossref reference lists
doi_to_paper = dict(zip(df['doi'].dropna(), df['pdf_file']))
G_citation = nx.DiGraph()

for _, row in df.iterrows():
    G_citation.add_node(row['pdf_file'])
    if pd.isna(row['crossref_references']):
        continue
    try:
        refs = ast.literal_eval(row['crossref_references'])
        for ref in refs:
            ref_doi = ref.get('doi')
            if ref_doi and ref_doi in doi_to_paper:
                G_citation.add_edge(row['pdf_file'], doi_to_paper[ref_doi])
    except Exception:
        continue

print("\n=== CITATION GRAPH ===")
print(f"Nodes: {G_citation.number_of_nodes()}")
print(f"Edges (internal citations): {G_citation.number_of_edges()}")
print(f"Isolated articles (0 connections): {len(list(nx.isolates(G_citation)))}")

# Build bibliographic coupling graph (shared Crossref references)
doi_cited_by = defaultdict(set)
for _, row in df.iterrows():
    if pd.isna(row['crossref_references']):
        continue
    try:
        refs = ast.literal_eval(row['crossref_references'])
        for ref in refs:
            if ref.get('doi'):
                doi_cited_by[ref['doi']].add(row['pdf_file'])
    except Exception:
        continue

G_coupling = nx.Graph()
for doi, papers in doi_cited_by.items():
    if len(papers) > 1:
        for p1, p2 in combinations(papers, 2):
            if G_coupling.has_edge(p1, p2):
                G_coupling[p1][p2]['weight'] += 1
            else:
                G_coupling.add_edge(p1, p2, weight=1)

print("\n=== BIBLIOGRAPHIC COUPLING ===")
print(f"Connected articles: {G_coupling.number_of_nodes()}")
print(f"Edges (shared references): {G_coupling.number_of_edges()}")
print(f"Isolated articles: {len(df) - G_coupling.number_of_nodes()}")

strong = [(u, v, d) for u, v, d in G_coupling.edges(data=True) if d['weight'] >= 5]
print(f"Strong connections (5+ shared refs): {len(strong)}")

print("\n=== SUMMARY ===")
coupling_coverage = G_coupling.number_of_nodes() / len(df) * 100
citation_coverage = G_citation.number_of_edges() / len(df) * 100
print(f"Bibliographic coupling covers: {coupling_coverage:.1f}% of articles")
print(f"Direct internal citations: {citation_coverage:.1f}% of corpus size")

if coupling_coverage > 50:
    print("Sufficient for full RQ2 analysis.")
elif coupling_coverage > 25:
    print("Moderate — RQ2 feasible with caveats.")
else:
    print("Low coverage — rely on keyword graph instead.")

Total articles: 2376
With DOI: 1552 (65.3%)
With references: 1552 (65.3%)

=== CITATION GRAPH ===
Nodes: 2376
Edges (internal citations): 0
Isolated articles (0 connections): 2376

=== BIBLIOGRAPHIC COUPLING ===
Connected articles: 2
Edges (shared references): 1
Isolated articles: 2374
Strong connections (5+ shared refs): 0

=== SUMMARY ===
Bibliographic coupling covers: 0.1% of articles
Direct internal citations: 0.0% of corpus size
Low coverage — rely on keyword graph instead.


## 2. Crossref Match Quality Filter

Restricts the corpus to articles with reliable Crossref matches (`title_similarity ≥ 0.85`) and counts how many Crossref-reported references resolve back to a corpus DOI.

In [4]:
meta = pd.read_csv(META_FINAL_CSV)

# Filter to articles with reliable Crossref matches
clean = meta[meta['title_similarity'] >= TITLE_SIM_THRESHOLD].copy()
print(f"Articles with reliable Crossref matching: {len(clean)}")

# Map DOI → filename for corpus articles
doi_to_fname = {}
for _, row in clean.iterrows():
    if isinstance(row['doi'], str):
        doi_to_fname[row['doi'].strip().lower()] = str(row['pdf_file']).replace('.pdf', '')

print(f"Unique DOIs in corpus: {len(doi_to_fname)}")

# Count potential citation edges resolvable via Crossref DOIs
found = 0
for _, row in clean.iterrows():
    refs_raw = row['crossref_references']
    if not isinstance(refs_raw, str):
        continue
    try:
        refs = ast.literal_eval(refs_raw)
    except Exception:
        continue
    for ref in refs:
        if isinstance(ref, dict):
            doi = ref.get('doi')
            if doi and doi.strip().lower() in doi_to_fname:
                found += 1

print(f"Potential citation edges (DOI in corpus): {found}")

Articles with reliable Crossref matching: 1226
Unique DOIs in corpus: 1091
Potential citation edges (DOI in corpus): 2


## 3. Title-Based Citation Edge Discovery

Attempts to find additional citation edges by exact-matching reference titles against Crossref titles in the corpus. `rapidfuzz` is imported for optional fuzzy matching.

In [ ]:
# Map normalized title → filename for corpus articles
title_to_fname = {}
for _, row in clean.iterrows():
    ct = row.get('crossref_title')
    if isinstance(ct, str):
        title_to_fname[ct.strip().lower()] = str(row['pdf_file']).replace('.pdf', '')

found_title = 0
for _, row in clean.iterrows():
    refs_raw = row['crossref_references']
    if not isinstance(refs_raw, str):
        continue
    try:
        refs = ast.literal_eval(refs_raw)
    except Exception:
        continue
    for ref in refs:
        if not isinstance(ref, dict):
            continue
        ref_title = ref.get('title')
        if not ref_title:
            continue
        ref_title_norm = ref_title.strip().lower()
        # Exact match; swap for fuzz.ratio(ref_title_norm, t) > 90 to enable fuzzy
        if ref_title_norm in title_to_fname:
            found_title += 1

print(f"Citation edges by title: {found_title}")

## 4. DOI Backfill via OpenAlex Title Search

For articles missing a DOI, searches OpenAlex by title (and optionally year + first author). Fills `doi` when similarity ≥ threshold; always stores the OpenAlex work ID for later lookups.

In [ ]:
SESSION.headers.update({"User-Agent": "leno4ka-doi-resolver/1", "Accept": "application/json"})


def _norm_doi(doi) -> str:
    """Normalize a DOI to lowercase without the doi.org prefix or URL query junk."""
    if doi is None or (isinstance(doi, float) and pd.isna(doi)):
        return ""
    s = str(doi).replace("https://doi.org/", "").strip().lower()
    return s.split("&", 1)[0].strip() if "&" in s else s


def _nonempty_doi(cell) -> bool:
    """Return True if cell contains a non-empty, non-NaN DOI-like string."""
    if cell is None or (isinstance(cell, float) and pd.isna(cell)):
        return False
    t = str(cell).strip()
    return bool(t) and t.lower() != "nan"


def _title_sim(a, b) -> float:
    """Return a 0–1 title similarity score (case-insensitive SequenceMatcher ratio)."""
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a.lower().strip(), b.lower().strip()).ratio()


def _oa_title(work) -> str:
    """Extract the title string from an OpenAlex work dict."""
    t = work.get("title") or ""
    if isinstance(t, list):
        t = t[0] if t else ""
    return str(t).strip()


def _oa_short_id(work_url: str) -> str:
    """Extract the short OpenAlex work ID (e.g. 'W12345') from a full URL."""
    if not work_url:
        return ""
    return work_url.rstrip("/").split("/")[-1]


def _parse_year(y) -> Optional[int]:
    """Return the four-digit publication year as int, or None if absent or unparseable."""
    if y is None or (isinstance(y, float) and pd.isna(y)):
        return None
    s = str(y).strip()
    if not s or s.lower() == "nan":
        return None
    m = re.search(r"(19|20)\d{2}", s)
    if m:
        return int(m.group(0))
    try:
        return int(float(s))
    except ValueError:
        return None


def _first_author_surname(authors_str) -> str:
    """Return the lowercased surname of the first semicolon-delimited author, or empty string."""
    if authors_str is None or (isinstance(authors_str, float) and pd.isna(authors_str)):
        return ""
    s = str(authors_str).strip()
    if not s:
        return ""
    first = re.split(r"[;,]", s, maxsplit=1)[0].strip()
    parts = first.replace(".", " ").split()
    if not parts:
        return ""
    cand = parts[-1].strip().lower()
    return cand if len(cand) > 1 else ""


def _oa_author_blob(work) -> str:
    """Return all author display names from an OpenAlex work as a single lowercase string."""
    names = []
    for a in (work.get("authorships") or [])[:12]:
        auth = (a or {}).get("author") or {}
        dn = auth.get("display_name")
        if dn:
            names.append(str(dn).lower())
    return " ".join(names)


def search_openalex_works(title: str, publication_year: Optional[int]) -> List[dict]:
    """Query OpenAlex for works matching title, optionally filtered by publication year."""
    search_q = (title or "")[:TITLE_SEARCH_MAX_LEN].strip()
    if len(search_q) < 8:
        return []
    params = {
        "search":   search_q,
        "per-page": SEARCH_PER_PAGE,
        "select":   "id,doi,title,publication_year,authorships",
    }
    if publication_year is not None:
        params["filter"] = f"publication_year:{publication_year}"
    r = SESSION.get("https://api.openalex.org/works", params=params, timeout=45)
    r.raise_for_status()
    return r.json().get("results") or []


def pick_best_work(
    row_title: str, row_year: Optional[int], row_authors: str, results: List[dict]
) -> tuple:
    """Select the best OpenAlex match by title similarity with optional author/year validation.

    Returns (work_dict, similarity_score). Returns (None, 0.0) if no acceptable match found.
    """
    best     = None
    best_sim = 0.0
    surname  = _first_author_surname(row_authors)
    for work in results:
        ty = work.get("publication_year")
        if row_year is not None and ty is not None:
            if abs(int(ty) - int(row_year)) > YEAR_SLACK:
                continue
        sim = _title_sim(row_title, _oa_title(work))
        if sim < best_sim:
            continue
        if surname:
            blob = _oa_author_blob(work)
            if surname not in blob and sim < 0.93:
                continue
        best_sim = sim
        best = work
    return best, best_sim


df = pd.read_csv(META_FINAL_CSV, low_memory=False)

for col in ("openalex_work_id", "openalex_matched_title", "openalex_match_similarity"):
    if col not in df.columns:
        df[col] = pd.NA

missing_mask = df["doi"].apply(lambda x: not _nonempty_doi(x))
has_title    = df["original_title"].apply(lambda x: isinstance(x, str) and len(str(x).strip()) >= 8)
to_try       = df.index[missing_mask & has_title]

print(f"Rows in table: {len(df)}")
print(f"Missing DOI: {missing_mask.sum()} | candidates (missing DOI + title): {len(to_try)}")

filled_doi = filled_oa_only = errors = 0

for idx in to_try:
    if _nonempty_doi(df.at[idx, "doi"]):
        continue
    if pd.notna(df.at[idx, "openalex_work_id"]) and str(df.at[idx, "openalex_work_id"]).strip():
        continue

    title   = str(df.at[idx, "original_title"]).strip()
    y       = _parse_year(df.at[idx, "original_year"])
    authors = df.at[idx, "original_authors"]

    try:
        results = search_openalex_works(title, y)
        time.sleep(REQUEST_SLEEP_S)
    except Exception as exc:
        errors += 1
        if errors <= 5:
            print(f"  API error row {idx}: {exc}")
        continue

    if not results:
        results = search_openalex_works(title, None)
        time.sleep(REQUEST_SLEEP_S)

    work, sim = pick_best_work(title, y, authors, results)
    if work is None or sim < TITLE_SIM_THRESHOLD:
        continue

    wid  = _oa_short_id(work.get("id", ""))
    ndoi = _norm_doi(work.get("doi") or "")

    df.at[idx, "openalex_work_id"]          = wid
    df.at[idx, "openalex_matched_title"]    = _oa_title(work)
    df.at[idx, "openalex_match_similarity"] = round(float(sim), 4)

    if ndoi:
        df.at[idx, "doi"] = ndoi
        filled_doi += 1
    else:
        filled_oa_only += 1

print(
    f"\nDone. Filled DOI from OpenAlex: {filled_doi} | "
    f"OpenAlex id only (no DOI on record): {filled_oa_only} | API errors: {errors}"
)
df.to_csv(META_OA_DOI_CSV, index=False)

with_doi = df["doi"].apply(_nonempty_doi).sum()
print(f"Rows with non-empty doi now: {with_doi} ({100 * with_doi / len(df):.1f}%)")

Rows in table: 2376
Missing DOI: 824 | candidates (missing DOI + title): 821
  API error row 323: 500 Server Error: Internal Server Error for url: https://api.openalex.org/works?search=POSITIONS+OF+VALUE+%2A2+IN+GENERALIZED+DOMINEERING+AND+CHESS&per-page=5&select=id%2Cdoi%2Ctitle%2Cpublication_year%2Cauthorships
  API error row 651: 400 Client Error: Bad Request for url: https://api.openalex.org/works?search=Welcome+to+Hamill+Indexing+%7C+home&per-page=5&select=id%2Cdoi%2Ctitle%2Cpublication_year%2Cauthorships
  API error row 1803: 500 Server Error: Internal Server Error for url: https://api.openalex.org/works?search=RESPONSE+TO+SIMON+AND+CHASE+%3C%2ARi%3E&per-page=5&select=id%2Cdoi%2Ctitle%2Cpublication_year%2Cauthorships&filter=publication_year%3A1973
  API error row 2040: 500 Server Error: Internal Server Error for url: https://api.openalex.org/works?search=Confidently+Selecting+a+Search+Heuristic+CONFIDENTL+Y+SELECTING+A+SEARCH+HEURISTIC%2A%29&per-page=5&select=id%2Cdoi%2Ctitle%2Cp

## 6. Fetch & Cache OpenAlex Referenced Works

Fetches `referenced_works` for every corpus article with a DOI and caches results to disk as JSON. On subsequent runs only missing DOIs are re-fetched.

In [ ]:
def _norm_id(pdf_name) -> str:
    """Return a lowercased article ID derived from the PDF filename (without .pdf extension)."""
    if pd.isna(pdf_name):
        return ""
    s = str(pdf_name).strip()
    return (s[:-4] if s.lower().endswith(".pdf") else s).lower()


def _norm_doi(doi) -> str:
    """Normalize a DOI to lowercase without the doi.org prefix or URL query junk."""
    if not isinstance(doi, str):
        return ""
    s = doi.replace("https://doi.org/", "").strip().lower()
    return s.split("&", 1)[0].strip() if "&" in s else s


def _pick_title_for_display(row) -> str:
    """Return the best available title: prefer Crossref, then original, then OpenAlex match."""
    for col in ("crossref_title", "original_title", "openalex_matched_title"):
        t = row.get(col)
        if pd.notna(t) and str(t).strip():
            return str(t).strip()
    pdf = row.get("pdf_file")
    if pd.notna(pdf) and str(pdf).strip():
        return str(pdf).strip()
    return str(row.get("article_id", ""))


def _fetch_batch(dois_batch: list) -> list:
    """Fetch OpenAlex work records (id, doi, referenced_works) for a batch of DOIs.

    Uses params= to safely encode special characters in DOIs (e.g. &).
    """
    r = requests.get(
        "https://api.openalex.org/works",
        params={
            "filter":   f"doi:{'|'.join(dois_batch)}",
            "select":   "id,doi,referenced_works",
            "per-page": len(dois_batch),
        },
        timeout=30,
    )
    r.raise_for_status()
    return r.json().get("results", [])


meta = pd.read_csv(META_OA_DOI_CSV)
meta["article_id"] = meta["pdf_file"].map(_norm_id)

doi_to_aid = {}
aid_to_doi = {}
for _, row in meta.iterrows():
    doi = _norm_doi(row.get("doi", ""))
    if doi:
        doi_to_aid[doi] = row["article_id"]
        aid_to_doi[row["article_id"]] = doi

corpus_dois = list(doi_to_aid.keys())
print(f"Corpus articles with DOI: {len(corpus_dois)}")

try:
    with open(CACHE_PATH) as f:
        cache = json.load(f)
    print(f"Cache loaded: {len(cache)} entries")
except FileNotFoundError:
    cache = {}
    print("No cache found — starting fresh")

to_fetch = [d for d in corpus_dois if d not in cache]
print(f"DOIs to fetch: {len(to_fetch)}")

n_batches = (len(to_fetch) + BATCH_SIZE - 1) // BATCH_SIZE
for i in range(0, len(to_fetch), BATCH_SIZE):
    batch = to_fetch[i : i + BATCH_SIZE]
    try:
        results = _fetch_batch(batch)
        for work in results:
            doi_norm = _norm_doi(work.get("doi", ""))
            if doi_norm:
                cache[doi_norm] = {
                    "openalex_id":      work.get("id", ""),
                    "referenced_works": work.get("referenced_works", []),
                }
        # Mark DOIs with no OpenAlex record to avoid refetching them
        found = {_norm_doi(w.get("doi", "")) for w in results}
        for doi in batch:
            if doi not in found and doi not in cache:
                cache[doi] = {"openalex_id": "", "referenced_works": []}
    except Exception as exc:
        print(f"  Batch {i // BATCH_SIZE + 1}/{n_batches} error: {exc}")

    if (i // BATCH_SIZE + 1) % 10 == 0 or i + BATCH_SIZE >= len(to_fetch):
        print(f"  Progress: {min(i + BATCH_SIZE, len(to_fetch))}/{len(to_fetch)}")

    time.sleep(0.12)

with open(CACHE_PATH, "w") as f:
    json.dump(cache, f)

mapped = sum(1 for d in corpus_dois if cache.get(d, {}).get("openalex_id"))
print(f"\nCache saved ({len(cache)} total). Corpus DOIs with OpenAlex record: {mapped}")

# Within-corpus citation edge summary
oa_id_to_aid_preview = {}
for doi, data in cache.items():
    if doi in doi_to_aid and data.get("openalex_id"):
        oa_id_to_aid_preview[data["openalex_id"]] = doi_to_aid[doi]

corpus_oa_ids_preview = set(oa_id_to_aid_preview)
citations_out = {}
citations_in  = {}

for doi, data in cache.items():
    if doi not in doi_to_aid:
        continue
    src = doi_to_aid[doi]
    hits = [
        oa_id_to_aid_preview[ref]
        for ref in data.get("referenced_works", [])
        if ref in corpus_oa_ids_preview and oa_id_to_aid_preview[ref] != src
    ]
    if hits:
        citations_out[src] = len(hits)
        for tgt in hits:
            citations_in[tgt] = citations_in.get(tgt, 0) + 1

total_edges     = sum(citations_out.values())
articles_citing = len(citations_out)
articles_cited  = len(citations_in)

print("\n── Citation edge summary ────────────────────────────────")
print(f"  Total within-corpus citation edges : {total_edges}")
print(f"  Articles that cite ≥1 corpus paper : {articles_citing}")
print(f"  Articles cited ≥1 time in corpus   : {articles_cited}")

print("\n  Top 10 most cited (in-corpus):")
for aid, cnt in sorted(citations_in.items(), key=lambda x: x[1], reverse=True)[:10]:
    title = next(
        (_pick_title_for_display(r) for _, r in meta[meta["article_id"] == aid].iterrows()),
        aid,
    )
    print(f"    {cnt:3d}×  {str(title)[:75]}")

print("\n  Top 10 articles citing the most corpus papers:")
for aid, cnt in sorted(citations_out.items(), key=lambda x: x[1], reverse=True)[:10]:
    title = next(
        (_pick_title_for_display(r) for _, r in meta[meta["article_id"] == aid].iterrows()),
        aid,
    )
    print(f"    {cnt:3d}→  {str(title)[:75]}")

Corpus articles with DOI: 1368
No cache found — starting fresh
DOIs to fetch: 1368
  Progress: 500/1368
  Progress: 1000/1368
  Progress: 1368/1368

Cache saved (1368 total). Corpus DOIs with OpenAlex record: 1356

── Citation edge summary ────────────────────────────────
  Total within-corpus citation edges : 3010
  Articles that cite ≥1 corpus paper : 717
  Articles cited ≥1 time in corpus   : 583

  Top 10 most cited (in-corpus):
     70×  XXII. Programming a computer for playing chess
     69×  Templates in Chess Memory: A Mechanism for Recalling Several Boards
     56×  A general reinforcement learning algorithm that masters chess, shogi, and G
     38×  A simulation of memory for chess positions
     36×  The history heuristic and alpha-beta search enhancements in practice
     35×  Expert memory: a comparison of four theories
     35×  The impact of chess research on cognitive science
     35×  Visuospatial abilities of chess players
     32×  Bandit Based Monte-Carlo Planning
 

## 7. Build Directed Citation Graph → GraphML + HTML

Reads the OpenAlex cache and builds a `networkx` directed graph of within-corpus citations. Exports to GraphML for downstream analysis and to an interactive Pyvis HTML for browsing.

In [ ]:
def _norm_id(pdf_name) -> str:
    """Return a lowercased article ID derived from the PDF filename (without .pdf extension)."""
    if pd.isna(pdf_name):
        return ""
    s = str(pdf_name).strip()
    return (s[:-4] if s.lower().endswith(".pdf") else s).lower()


def _norm_doi(doi) -> str:
    """Normalize a DOI to lowercase without the doi.org prefix."""
    if not isinstance(doi, str):
        return ""
    return doi.replace("https://doi.org/", "").strip().lower()


def _pick_title(row) -> str:
    """Return the best available title: prefer Crossref, then original, then OpenAlex match."""
    for col in ("crossref_title", "original_title", "openalex_matched_title"):
        t = row.get(col)
        if pd.notna(t) and str(t).strip():
            return str(t).strip()
    pdf = row.get("pdf_file")
    if pd.notna(pdf) and str(pdf).strip():
        return str(pdf).strip()
    return str(row.get("article_id", "unknown"))


def _pick_authors(row) -> list:
    """Return a list of author name strings from Crossref or original author fields."""
    for col in ("crossref_authors", "original_authors"):
        a = row.get(col)
        if isinstance(a, str) and a.strip():
            return [p.strip() for p in a.split(";") if p.strip()]
    return []


meta = pd.read_csv(META_OA_YEAR_CSV)
meta["article_id"] = meta["pdf_file"].map(_norm_id)

with open(CACHE_PATH) as f:
    cache = json.load(f)

doi_to_aid = {}
aid_to_doi = {}
for _, row in meta.iterrows():
    doi = _norm_doi(row.get("doi", ""))
    if doi:
        doi_to_aid[doi] = row["article_id"]
        aid_to_doi[row["article_id"]] = doi

oa_id_to_aid = {}
for doi, data in cache.items():
    if doi in doi_to_aid and data.get("openalex_id"):
        oa_id_to_aid[data["openalex_id"]] = doi_to_aid[doi]

corpus_oa_ids = set(oa_id_to_aid)
print(f"Corpus articles with OpenAlex ID: {len(corpus_oa_ids)}")

# Build directed citation graph
G = nx.DiGraph()

for _, row in meta.iterrows():
    aid = row["article_id"]
    G.add_node(
        aid,
        label   =_pick_title(row),
        title   =_pick_title(row),
        pdf_file=str(row["pdf_file"]),
        authors ="|".join(_pick_authors(row)),
        year    =str(row.get("crossref_year") or row.get("original_year") or row.get("openalex_year") or ""),
        doi     =aid_to_doi.get(aid, ""),
    )

for doi, data in cache.items():
    if doi not in doi_to_aid:
        continue
    src = doi_to_aid[doi]
    for ref_oa_id in data.get("referenced_works", []):
        if ref_oa_id in corpus_oa_ids:
            tgt = oa_id_to_aid[ref_oa_id]
            if src != tgt:
                G.add_edge(src, tgt, edge_type="citation")

print(f"Nodes: {G.number_of_nodes()} | Citation edges: {G.number_of_edges()}")

in_deg    = dict(G.in_degree())
top_cited = sorted(in_deg.items(), key=lambda x: x[1], reverse=True)[:10]
print("\nTop 10 most cited within corpus:")
for aid, deg in top_cited:
    print(f"  {deg:3d}×  {G.nodes[aid].get('label', aid)[:80]}")

components = list(nx.weakly_connected_components(G))
components.sort(key=len, reverse=True)
print(f"\nWeakly connected components: {len(components)}")
print(f"Largest component: {len(components[0])} nodes")

nx.write_graphml(G, GRAPHML_OUT)
print(f"\nGraphML → {GRAPHML_OUT}")

# Pyvis visualization (nodes with ≥1 edge only)
G_loaded = nx.read_graphml(GRAPHML_OUT)
nodes_with_edges = {n for n, d in G_loaded.degree() if d > 0}
G_vis = G_loaded.subgraph(nodes_with_edges).copy()
print(f"Visualizing {G_vis.number_of_nodes()} nodes, {G_vis.number_of_edges()} edges")

net = Network(
    height="100vh", width="100%",
    bgcolor="#111111", font_color="white",
    notebook=False, directed=True,
)
net.toggle_physics(True)
net.set_options("""{
  "nodes": {
    "scaling": {"min": 6, "max": 28},
    "font": {"size": 11}
  },
  "physics": {
    "enabled": true,
    "solver": "forceAtlas2Based",
    "forceAtlas2Based": {
      "theta": 0.5,
      "gravitationalConstant": -280,
      "centralGravity": 0.001,
      "springLength": 420,
      "springConstant": 0.018,
      "damping": 0.52,
      "avoidOverlap": 0.92
    },
    "minVelocity": 0.4,
    "maxVelocity": 42,
    "timestep": 0.35,
    "stabilization": {
      "enabled": true,
      "iterations": 900,
      "updateInterval": 20,
      "fit": true
    }
  },
  "edges": {
    "length": 380,
    "arrows": {"to": {"enabled": true, "scaleFactor": 0.6}},
    "smooth": {"type": "curvedCW", "roundness": 0.2},
    "color": {"color": "#e74c3c", "opacity": 0.7}
  },
  "interaction": {
    "hover": true,
    "navigationButtons": true,
    "tooltipDelay": 120
  }
}""")

for n, nd in G_vis.nodes(data=True):
    in_d  = G_vis.in_degree(n)
    out_d = G_vis.out_degree(n)
    label = nd.get("label", n)
    net.add_node(
        n,
        label=label[:65],
        title=(
            f"<b>{label}</b><br>"
            f"Authors: {nd.get('authors','')}<br>"
            f"Year: {nd.get('year','')}<br>"
            f"DOI: {nd.get('doi','')}<br>"
            f"Cited by (in corpus): {in_d} | Cites (in corpus): {out_d}"
        ),
        value=max(1, 2 + in_d * 1.25),
    )

for u, v in G_vis.edges():
    net.add_edge(u, v, color="#e74c3c", width=1.0, title="cites")

net.write_html(str(HTML_OUT), open_browser=False)
print(f"HTML  → {HTML_OUT}")
print("Open the HTML file in your browser.")

Corpus articles with OpenAlex ID: 1356
Nodes: 2376 | Citation edges: 3010

Top 10 most cited within corpus:
   70×  XXII. Programming a computer for playing chess
   69×  Templates in Chess Memory: A Mechanism for Recalling Several Boards
   56×  A general reinforcement learning algorithm that masters chess, shogi, and Go thr
   38×  A simulation of memory for chess positions
   36×  The history heuristic and alpha-beta search enhancements in practice
   35×  Visuospatial abilities of chess players
   35×  The impact of chess research on cognitive science
   35×  Expert memory: a comparison of four theories
   32×  Bandit Based Monte-Carlo Planning
   30×  The role of deliberate practice in chess expertise

Weakly connected components: 1508
Largest component: 850 nodes

GraphML → ../output/articles_citations.graphml
Visualizing 883 nodes, 3010 edges
HTML  → ../output/articles_citations.html
Open the HTML file in your browser.

## 8. Era-Tagged Visualization (RQ4)

Enriches the base citation graph with publication-era labels, arranges nodes in a left-to-right timeline strip (one column per era), and exports a static Pyvis HTML.

In [ ]:
def get_era(year) -> str:
    """Return the publication-era bin label for a given year value."""
    if pd.isna(year) or year == "" or str(year).strip().lower() == "nan":
        return "unknown_year"
    try:
        year = int(float(str(year).strip()))
    except (TypeError, ValueError):
        return "unknown_year"
    if 1950 <= year < 1977:
        return "1950-1977"
    if 1977 <= year < 1997:
        return "1977-1997"
    if 1997 <= year < 2017:
        return "1997-2017"
    if year >= 2017:
        return "2017+"
    return "before_1950"


def _norm_id(pdf_name) -> str:
    """Return a lowercased article ID derived from the PDF filename (without .pdf extension)."""
    if pd.isna(pdf_name):
        return ""
    s = str(pdf_name).strip()
    return (s[:-4] if s.lower().endswith(".pdf") else s).lower()


if not Path(GRAPHML_OUT).is_file():
    raise FileNotFoundError(f"Run the build-graph cell first — missing {GRAPHML_OUT}")
G = nx.read_graphml(GRAPHML_OUT)
print(f"Loaded base GraphML: {GRAPHML_OUT} ({G.number_of_nodes()} nodes, {G.number_of_edges()} edges)")

meta = pd.read_csv(META_OA_YEAR_CSV)
meta["article_id"] = meta["pdf_file"].map(_norm_id)
aid_to_year = {}
for _, row in meta.iterrows():
    aid = row["article_id"]
    y = row.get("crossref_year")
    if pd.isna(y) or str(y).strip() == "":
        y = row.get("original_year")
    if pd.isna(y) or str(y).strip() == "":
        y = row.get("openalex_year")
    aid_to_year[aid] = y

ERA_COLORS = {
    "before_1950":  "#9b59b6",
    "1950-1977":    "#3498db",
    "1977-1997":    "#1abc9c",
    "1997-2017":    "#f39c12",
    "2017+":        "#e74c3c",
    "unknown_year": "#7f8c8d",
}

for n in G.nodes():
    y_raw = G.nodes[n].get("year", "") or ""
    if (not str(y_raw).strip() or str(y_raw).strip().lower() == "nan") and n in aid_to_year:
        y_raw = aid_to_year.get(n, "")
    G.nodes[n]["era"] = get_era(y_raw)

nx.write_graphml(G, GRAPHML_ERA_OUT)
print(f"GraphML (era attributes) → {GRAPHML_ERA_OUT}")

era_counts = {}
for n in G.nodes():
    e = G.nodes[n].get("era", "unknown_year")
    era_counts[e] = era_counts.get(e, 0) + 1
print("Nodes per era:", dict(sorted(era_counts.items(), key=lambda x: x[0])))

ERA_ORDER = ["before_1950", "1950-1977", "1977-1997", "1997-2017", "2017+", "unknown_year"]

nodes_with_edges = {n for n, d in G.degree() if d > 0}
G_vis = G.subgraph(nodes_with_edges).copy()

eras_in_graph = {G_vis.nodes[n].get("era", "unknown_year") for n in G_vis.nodes()}
eras_ranked   = sorted(
    eras_in_graph,
    key=lambda e: (ERA_ORDER.index(e) if e in ERA_ORDER else 99, e),
)

n_era = len(eras_ranked)
sep_x = 2400
era_hub = {}
for i, era in enumerate(eras_ranked):
    x0 = (i - 0.5 * (n_era - 1)) * sep_x if n_era > 1 else 0.0
    era_hub[era] = (x0, 0.0)

by_era = defaultdict(list)
for n in G_vis.nodes():
    by_era[G_vis.nodes[n].get("era", "unknown_year")].append(n)

GOLDEN    = 2.39996322972865332
positions = {}
for era, nlist in by_era.items():
    cx, cy  = era_hub[era]
    m       = len(nlist)
    cloud_r = max(260.0, 38.0 * math.sqrt(m))
    for j, nid in enumerate(nlist):
        t   = math.sqrt((j + 1) / max(m, 1))
        rad = cloud_r * t * 0.95
        ga  = j * GOLDEN
        positions[nid] = (cx + rad * math.cos(ga), cy + rad * math.sin(ga) * 0.85)

net = Network(
    height="100vh", width="100%",
    bgcolor="#111111", font_color="white",
    notebook=False, directed=True,
)
net.toggle_physics(False)
net.set_options("""{
  "nodes": {
    "scaling": {"min": 5, "max": 22},
    "font": {"size": 10, "strokeWidth": 0}
  },
  "physics": {"enabled": false},
  "edges": {
    "length": 80,
    "arrows": {"to": {"enabled": true, "scaleFactor": 0.35}},
    "smooth": false,
    "color": {"color": "#888888", "opacity": 0.14},
    "selectionWidth": 1
  },
  "interaction": {
    "hover": true,
    "navigationButtons": true,
    "tooltipDelay": 120
  }
}""")

for n, nd in G_vis.nodes(data=True):
    in_d  = G_vis.in_degree(n)
    out_d = G_vis.out_degree(n)
    label = nd.get("label", n)
    era   = nd.get("era", "unknown_year")
    x, y  = positions[n]
    yr    = nd.get("year", "") or ""
    if str(yr).strip().lower() in ("", "nan", "none"):
        yr = ""
    net.add_node(
        n,
        x=x, y=y, physics=False,
        label=label[:65],
        title=(
            f"<b>{label}</b><br>"
            f"Era: {era}<br>"
            f"Authors: {nd.get('authors', '')}<br>"
            f"Year: {yr}<br>"
            f"DOI: {nd.get('doi', '')}<br>"
            f"Cited by (in corpus): {in_d} | Cites (in corpus): {out_d}"
        ),
        value=max(1, 2 + in_d * 1.25),
        color=ERA_COLORS.get(era, "#7f8c8d"),
        group=era,
    )

for u, v in G_vis.edges():
    net.add_edge(u, v, color="#888888", width=0.28, title="cites")

net.write_html(str(HTML_ERA_OUT), open_browser=False)

# vis-network does not auto-fit when physics is off — inject a one-shot fit callback
html    = Path(HTML_ERA_OUT).read_text(encoding="utf-8")
_needle = "network = new vis.Network(container, data, options);"
_fit_js = """
                  setTimeout(function () {
                    try { network.fit({ padding: 100, animation: { duration: 0, easingFunction: "linear" } }); } catch (e) {}
                  }, 80);
"""
if _needle in html and _fit_js.strip() not in html:
    html = html.replace(_needle, _needle + _fit_js, 1)
    Path(HTML_ERA_OUT).write_text(html, encoding="utf-8")

print(f"HTML (era strip + faint edges + auto-fit) → {HTML_ERA_OUT}")
print("Open the HTML file in your browser.")

Loaded base GraphML: ../output/articles_citations.graphml (2376 nodes, 3010 edges)
GraphML (era attributes) → ../output/articles_citations_by_era.graphml
Nodes per era: {'1950-1977': 46, '1977-1997': 248, '1997-2017': 1106, '2017+': 555, 'before_1950': 12, 'unknown_year': 409}
HTML (era strip + faint edges + auto-fit) → ../output/articles_citations_by_era.html
Open the HTML file in your browser.

## 9. Backfill Missing Publication Years

Fetches `publication_year` from OpenAlex for articles with neither a Crossref nor an original year. Uses DOI-cache OpenAlex IDs first, then `openalex_work_id` from the DOI backfill step.

In [ ]:
def _norm_doi(doi) -> str:
    """Normalize a DOI to lowercase without the doi.org prefix or URL query junk."""
    if not isinstance(doi, str):
        return ""
    s = doi.replace("https://doi.org/", "").strip().lower()
    return s.split("&", 1)[0].strip() if "&" in s else s


def _nonempty(cell) -> bool:
    """Return True if cell is a non-empty, non-NaN value."""
    if cell is None or (isinstance(cell, float) and pd.isna(cell)):
        return False
    return bool(str(cell).strip()) and str(cell).strip().lower() != "nan"


def _parse_year(y) -> int | None:
    """Return the four-digit publication year as int, or None if absent or unparseable."""
    if not _nonempty(y):
        return None
    m = re.search(r"(19|20)\d{2}", str(y))
    return int(m.group()) if m else None


def _short_id(oa_url: str) -> str:
    """Extract the short OpenAlex work ID (e.g. 'W12345') from a full URL."""
    return str(oa_url).rstrip("/").split("/")[-1] if oa_url else ""


df = pd.read_csv(META_OA_DOI_CSV, low_memory=False)
if "openalex_year" not in df.columns:
    df["openalex_year"] = pd.NA

needs_year = df.index[
    df["crossref_year"].apply(lambda x: _parse_year(x) is None) &
    df["original_year"].apply(lambda x: _parse_year(x) is None)
]
print(f"Rows total: {len(df)}")
print(f"Rows missing year (crossref + original): {len(needs_year)}")

with open(CACHE_PATH) as f:
    cache = json.load(f)

doi_to_short = {
    doi: _short_id(data.get("openalex_id", ""))
    for doi, data in cache.items()
    if data.get("openalex_id")
}

idx_to_short = {}
for idx in needs_year:
    # Source 1: DOI present in cache
    doi = _norm_doi(df.at[idx, "doi"])
    if doi and doi in doi_to_short:
        idx_to_short[idx] = doi_to_short[doi]
        continue
    # Source 2: openalex_work_id from DOI-resolver (no-DOI rows)
    wid = df.at[idx, "openalex_work_id"] if "openalex_work_id" in df.columns else None
    if _nonempty(wid):
        idx_to_short[idx] = _short_id(str(wid))

print(f"Rows with OpenAlex ID to query: {len(idx_to_short)}")
print(f"Rows with no OpenAlex coverage: {len(needs_year) - len(idx_to_short)}")

short_id_to_year = {}
short_ids = list(set(idx_to_short.values()))

for i in range(0, len(short_ids), BATCH_SIZE):
    batch = short_ids[i : i + BATCH_SIZE]
    try:
        r = requests.get(
            "https://api.openalex.org/works",
            params={
                "filter":   f"openalex_id:{'|'.join(batch)}",
                "select":   "id,publication_year",
                "per-page": BATCH_SIZE,
            },
            timeout=30,
        )
        r.raise_for_status()
        for work in r.json().get("results", []):
            sid = _short_id(work.get("id", ""))
            yr  = work.get("publication_year")
            if sid and yr is not None:
                short_id_to_year[sid] = int(yr)
    except Exception as exc:
        print(f"  Batch {i // BATCH_SIZE + 1} error: {exc}")
    time.sleep(REQUEST_SLEEP_S)

print(f"Years fetched from OpenAlex: {len(short_id_to_year)}")

filled = 0
for idx, sid in idx_to_short.items():
    yr = short_id_to_year.get(sid)
    if yr is not None:
        df.at[idx, "openalex_year"] = yr
        filled += 1

print(f"\nopenalex_year filled for: {filled} rows")


def _has_year(row) -> bool:
    """Return True if the row has at least one non-null year source."""
    return any(
        _parse_year(row[c]) is not None
        for c in ("crossref_year", "original_year", "openalex_year")
        if c in row.index
    )


covered = df.apply(_has_year, axis=1).sum()
print(f"Rows with at least one year source: {covered} / {len(df)} ({100 * covered / len(df):.1f}%)")
print(f"Still missing year: {len(df) - covered} rows")

df.to_csv(META_OA_YEAR_CSV, index=False)
print(f"\nSaved → {META_OA_YEAR_CSV}")

Rows total: 2376
Rows missing year (crossref + original): 677
Rows with OpenAlex ID to query: 267
Rows with no OpenAlex coverage: 410
Years fetched from OpenAlex: 251

openalex_year filled for: 267 rows
Rows with at least one year source: 1966 / 2376 (82.7%)
Still missing year: 410 rows

Saved → ../output/metadata_enriched_after_openalex_year.csv